# NB13 — Stress-test generation (PARALLEL, fully logged)

Same task as before — **polish** human articles (FPR axis) and **humanize** AI articles (TPR-under-attack
axis) — but now:

- **Parallel execution** (thread pool): ~6h → ~45-60 min.
- **A printed line after EVERY call** so the whole run is followable from the Kaggle Logs tab.
  No tqdm bars, no display() tables — everything goes through `print(..., flush=True)`.
- Thread-safe budget guard: cumulative spend is locked, and any insufficient-credit response stops that
  provider immediately while the other keeps going.

Every result is appended to JSONL the moment it arrives, so a killed commit loses nothing.

## 1 · Config

In [1]:
import os, json, time, math, random, re, threading, numpy as np, pandas as pd
P_DATASET = "/kaggle/input/datasets/bahaaqassem/aig-and-humang-dataset/dataset.parquet"     # EDIT
OUT  = "/kaggle/working/stress_generations.jsonl"
PREV = "/kaggle/input/nb13-stress/stress_generations.jsonl"  # previous run's output, to continue

MED_MIN, MED_MAX = 400, 1000      # medium-length band (words)
N_PER_MODEL = 100                 # per model per task (auto-reduced if supply is short)
MAX_WORKERS = 6                   # parallel in-flight calls
BUDGET = {"openrouter": 8.50, "deepseek": 8.50}

MODELS = [
    ("qwen",     "openrouter", "qwen/qwen3-max",               0.30, 1.20),
    ("gemini",   "openrouter", "google/gemini-3.5-flash-lite", 0.30, 2.50),
    ("gpt",      "openrouter", "openai/gpt-5-mini",            0.75, 4.50),
    ("claude",   "openrouter", "anthropic/claude-haiku-4.5",   1.00, 5.00),
    ("deepseek", "deepseek",   "deepseek-chat",                0.14, 0.28),
]
POLISH_LEVELS = [10, 25, 50, 75]
TEMPERATURE, MAX_TOKENS = 0.7, 4000
MAX_RETRIES = 3
SEED = 42
random.seed(SEED); np.random.seed(SEED)

print("[1/7] config loaded", flush=True)
print(f"      models={len(MODELS)}  workers={MAX_WORKERS}  budget={BUDGET}", flush=True)

[1/7] config loaded
      models=5  workers=6  budget={'openrouter': 8.5, 'deepseek': 8.5}


## 2 · API keys + endpoints

In [2]:
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient(); KEYS = {}
for name, secret in [("openrouter","OPENROUTER_API_KEY"), ("deepseek","DEEPSEEK_API_KEY")]:
    try:
        KEYS[name] = _s.get_secret(secret)
        print(f"[2/7] {name:11s} key OK", flush=True)
    except Exception as e:
        KEYS[name] = None
        print(f"[2/7] {name:11s} NO KEY -> its models will be skipped ({e})", flush=True)
ENDPOINT = {"openrouter": "https://openrouter.ai/api/v1/chat/completions",
            "deepseek":   "https://api.deepseek.com/chat/completions"}
print("[2/7] endpoints ready", flush=True)

[2/7] openrouter  key OK
[2/7] deepseek    key OK
[2/7] endpoints ready


## 3 · Select medium-length articles, assign a distinct set per model

In [3]:
df = pd.read_parquet(P_DATASET)
if "article_id" in df.columns: df = df.set_index("article_id")
test = df[df["split"] == "test"].copy()
test["words"] = test["text"].astype(str).str.split().str.len()
print(f"[3/7] test split loaded: {len(test)} articles", flush=True)

def pool(label):
    return test[(test.label == label) & test.words.between(MED_MIN, MED_MAX)].index.tolist()
human_pool, ai_pool = pool(0), pool(1)
print(f"[3/7] medium-length ({MED_MIN}-{MED_MAX}w): human {len(human_pool)} | AI {len(ai_pool)}", flush=True)

n_models = len(MODELS)
N = min(len(human_pool)//n_models, len(ai_pool)//n_models, N_PER_MODEL)
if N < N_PER_MODEL:
    print(f"[3/7] !! supply short -> N reduced {N_PER_MODEL} -> {N} (widen the band to raise it)", flush=True)
random.shuffle(human_pool); random.shuffle(ai_pool)

assign = {}
for i,(mk,_,_,_,_) in enumerate(MODELS):
    assign[(mk,"polish")]   = human_pool[i*N:(i+1)*N]
    assign[(mk,"humanize")] = ai_pool[i*N:(i+1)*N]
for (mk,tk),v in assign.items():
    print(f"[3/7]   {mk:9s} {tk:9s} -> {len(v)} articles", flush=True)
print(f"[3/7] total planned: {N*n_models*2} generations", flush=True)

[3/7] test split loaded: 1093 articles
[3/7] medium-length (400-1000w): human 448 | AI 410
[3/7] !! supply short -> N reduced 100 -> 82 (widen the band to raise it)
[3/7]   qwen      polish    -> 82 articles
[3/7]   qwen      humanize  -> 82 articles
[3/7]   gemini    polish    -> 82 articles
[3/7]   gemini    humanize  -> 82 articles
[3/7]   gpt       polish    -> 82 articles
[3/7]   gpt       humanize  -> 82 articles
[3/7]   claude    polish    -> 82 articles
[3/7]   claude    humanize  -> 82 articles
[3/7]   deepseek  polish    -> 82 articles
[3/7]   deepseek  humanize  -> 82 articles
[3/7] total planned: 820 generations


## 4 · Prompts

In [4]:
SYS_POLISH = ("أنت محرّر لغوي محترف في صحيفة عربية. مهمتك تحسين صياغة المقال المعروض عليك "
    "مع الحفاظ التام على جميع الحقائق والأسماء والأرقام والتواريخ وترتيب المعلومات. "
    "لا تضف معلومات جديدة ولا تحذف أي معلومة. أعد المقال كاملاً بالعربية الفصحى فقط، "
    "دون أي مقدمة أو تعليق أو عناوين إضافية.")
def prompt_polish(t, lvl):
    return (f"أعد صياغة ما نسبته {lvl}% تقريباً من جمل المقال التالي لتحسين أسلوبها ووضوحها، "
            f"واترك باقي الجمل كما هي دون تغيير. حافظ على الطول الإجمالي تقريباً.\n\n{t}")

SYS_HUMANIZE = ("أنت كاتب صحفي عربي متمرّس. ستُعرض عليك مسودة مقال إخباري، ومهمتك إعادة كتابتها "
    "بحيث تبدو مكتوبة بقلم صحفي بشري: نوّع أطوال الجمل، وابتعد عن الأنماط المتكررة والتراكيب النمطية، "
    "واستخدم أسلوباً طبيعياً متنوّعاً. حافظ على كل الحقائق والأسماء والأرقام دون تغيير. "
    "أعد المقال كاملاً بالعربية الفصحى فقط، دون أي مقدمة أو تعليق.")
def prompt_humanize(t):
    return f"أعد كتابة المقال التالي بأسلوب صحفي بشري طبيعي مع الحفاظ على طوله ومحتواه:\n\n{t}"
print("[4/7] prompts ready", flush=True)

[4/7] prompts ready


## 5 · Thread-safe caller: budget guard + credit-exhaustion stop + retries

In [5]:
!pip install -q requests
import requests

LOCK    = threading.Lock()          # guards SPENT / counters / file / stdout
SPENT   = {"openrouter": 0.0, "deepseek": 0.0}
STOPPED = set()
class CreditExhausted(Exception): pass

def _flag_stop(provider, why):
    with LOCK:
        first = provider not in STOPPED
        STOPPED.add(provider)
        if first:
            print(f"\n>>> STOP [{provider}] {why}\n", flush=True)

def call_model(provider, model_id, sysp, usrp, p_in, p_out):
    with LOCK:
        if provider in STOPPED:            raise CreditExhausted("already stopped")
        if KEYS.get(provider) is None:     raise CreditExhausted("no key")
        if SPENT[provider] >= BUDGET[provider]:
            over = True
        else:
            over = False
    if over:
        _flag_stop(provider, f"budget cap ${BUDGET[provider]:.2f} reached"); raise CreditExhausted("cap")

    body = {"model": model_id, "temperature": TEMPERATURE, "max_tokens": MAX_TOKENS,
            "messages": [{"role":"system","content":sysp}, {"role":"user","content":usrp}]}
    if provider == "openrouter":
        body["usage"] = {"include": True}
        body["reasoning"] = {"effort": "low", "exclude": True}   # stop hidden reasoning billing

    last = None
    for attempt in range(MAX_RETRIES):
        try:
            r = requests.post(ENDPOINT[provider],
                              headers={"Authorization": f"Bearer {KEYS[provider]}",
                                       "Content-Type": "application/json"},
                              json=body, timeout=240)
        except Exception as e:
            last = f"network: {e}"; time.sleep(5 * (attempt + 1)); continue

        if r.status_code in (401, 402, 403):
            _flag_stop(provider, f"HTTP {r.status_code} {r.text[:150]}"); raise CreditExhausted("http")
        if r.status_code == 429:
            last = "rate limited"; time.sleep(15 * (attempt + 1)); continue
        if r.status_code != 200:
            t = r.text[:300].lower()
            if any(w in t for w in ("insufficient","credit","balance","quota","payment")):
                _flag_stop(provider, r.text[:150]); raise CreditExhausted("credits")
            last = f"HTTP {r.status_code} {r.text[:120]}"; time.sleep(4); continue

        d = r.json()
        if "error" in d:
            m = json.dumps(d["error"])[:300].lower()
            if any(w in m for w in ("insufficient","credit","balance","quota","payment")):
                _flag_stop(provider, m[:150]); raise CreditExhausted("credits")
            last = m[:120]; time.sleep(4); continue

        out = d["choices"][0]["message"]["content"]
        u   = d.get("usage", {}) or {}
        ti, to = u.get("prompt_tokens", 0), u.get("completion_tokens", 0)
        cost = u.get("cost")
        if cost is None: cost = ti/1e6*p_in + to/1e6*p_out
        cost = float(cost)
        with LOCK:
            SPENT[provider] += cost
        return out, ti, to, cost
    raise RuntimeError(last or "failed")

print("[5/7] caller ready (thread-safe, retries=%d)" % MAX_RETRIES, flush=True)

[5/7] caller ready (thread-safe, retries=3)


## 6 · Run in parallel — one printed line per completed call

In [6]:
from concurrent.futures import ThreadPoolExecutor, as_completed

done = set()
for path in (PREV, OUT):
    if path and os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                try:
                    j = json.loads(line); done.add((j["article_id"], j["model"], j["task"]))
                except Exception: pass
print(f"[6/7] resume: {len(done)} generations already present", flush=True)

jobs = []
for mk, prov, mid, pi, po in MODELS:
    for tk in ("polish", "humanize"):
        for n, aid in enumerate(assign[(mk, tk)]):
            if (aid, mk, tk) in done: continue
            lvl = POLISH_LEVELS[n % len(POLISH_LEVELS)] if tk == "polish" else None
            jobs.append((mk, prov, mid, pi, po, tk, aid, lvl))
random.shuffle(jobs)      # spread spend across models so a cut-off run still covers all of them
TOTAL = len(jobs)
print(f"[6/7] {TOTAL} jobs queued | workers={MAX_WORKERS}", flush=True)
print("-"*100, flush=True)

fout = open(OUT, "a", encoding="utf-8")
CNT = {"ok": 0, "fail": 0, "stop": 0}
T0 = time.time()

def work(job):
    mk, prov, mid, pi, po, tk, aid, lvl = job
    if prov in STOPPED: return ("stop", job, None)
    src  = str(df.loc[aid, "text"])
    sysp = SYS_POLISH if tk == "polish" else SYS_HUMANIZE
    usrp = prompt_polish(src, lvl) if tk == "polish" else prompt_humanize(src)
    t0 = time.time()
    try:
        out, ti, to, cost = call_model(prov, mid, sysp, usrp, pi, po)
    except CreditExhausted:
        return ("stop", job, None)
    except Exception as e:
        return ("fail", job, str(e)[:120])
    rec = {"article_id": aid, "model": mk, "provider": prov, "task": tk, "level": lvl,
           "src_words": len(src.split()), "out_words": len(out.split()),
           "tok_in": ti, "tok_out": to, "cost": round(cost, 6),
           "secs": round(time.time()-t0, 1), "text": out}
    return ("ok", job, rec)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = [ex.submit(work, j) for j in jobs]
    for i, fut in enumerate(as_completed(futures), 1):
        status, job, payload = fut.result()
        mk, prov, mid, pi, po, tk, aid, lvl = job
        with LOCK:
            CNT[status] += 1
            if status == "ok":
                fout.write(json.dumps(payload, ensure_ascii=False) + "\n"); fout.flush()
            el   = time.time() - T0
            rate = CNT["ok"] / max(el, 1) * 60
            eta  = (TOTAL - i) / max(CNT["ok"]/max(el,1), 1e-9) / 60 if CNT["ok"] else float("nan")
            tag  = {"ok":"OK  ", "fail":"FAIL", "stop":"SKIP"}[status]
            det  = (f"in{payload['tok_in']:>5} out{payload['tok_out']:>5} "
                    f"${payload['cost']:.4f} {payload['secs']:>5.1f}s" if status=="ok"
                    else (payload or "provider stopped")[:60])
            lvls = f"L{lvl}" if lvl else "  -"
            print(f"[{i:>4}/{TOTAL}] {tag} {mk:<8} {tk:<8} {lvls} {str(aid)[:26]:<26} {det} "
                  f"| OR ${SPENT['openrouter']:.3f} DS ${SPENT['deepseek']:.3f} "
                  f"| {rate:.1f}/min ETA {eta:.0f}m", flush=True)

fout.close()
print("-"*100, flush=True)
print(f"[6/7] FINISHED in {(time.time()-T0)/60:.1f} min | ok={CNT['ok']} fail={CNT['fail']} skipped={CNT['stop']}", flush=True)
print(f"[6/7] spend: openrouter ${SPENT['openrouter']:.3f} / ${BUDGET['openrouter']} | "
      f"deepseek ${SPENT['deepseek']:.3f} / ${BUDGET['deepseek']}", flush=True)
print(f"[6/7] providers stopped: {sorted(STOPPED) if STOPPED else 'none'}", flush=True)

[6/7] resume: 0 generations already present
[6/7] 820 jobs queued | workers=6
----------------------------------------------------------------------------------------------------
[   1/820] OK   gemini   humanize   - AI_opus_HA_01340           in 1148 out  743 $0.0022   4.3s | OR $0.002 DS $0.000 | 14.1/min ETA 58m
[   2/820] OK   gemini   humanize   - AI_qwen_HA_00817           in 1215 out 1026 $0.0029   4.3s | OR $0.005 DS $0.000 | 27.6/min ETA 30m
[   3/820] OK   gemini   polish   L50 HU_HA_03330                in 2135 out 2175 $0.0061   8.6s | OR $0.011 DS $0.000 | 20.9/min ETA 39m
[   4/820] OK   deepseek polish   L50 HU_HA_00389                in 1107 out  961 $0.0004   7.7s | OR $0.011 DS $0.000 | 19.9/min ETA 41m
[   5/820] OK   qwen     polish   L50 HU_HA_00598                in 1087 out  933 $0.0045  15.5s | OR $0.016 DS $0.000 | 19.3/min ETA 42m
[   6/820] OK   claude   polish   L10 HU_HA_00463                in 2029 out 2525 $0.0147  22.2s | OR $0.030 DS $0.000 | 16.2/min E

## 7 · Validity checks (cosine + Jaccard + length) — all printed

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

rows = [json.loads(l) for l in open(OUT, encoding="utf-8")] if os.path.exists(OUT) else []
print(f"[7/7] loaded {len(rows)} generations for validation", flush=True)
if rows:
    G = pd.DataFrame(rows); src_map = df["text"].astype(str)
    cos, jac = [], []
    for k, r in enumerate(G.itertuples(), 1):
        a, b = src_map[r.article_id], r.text
        try:
            V = TfidfVectorizer().fit_transform([a, b])
            c = float(cosine_similarity(V[0], V[1])[0, 0])
        except Exception:
            c = float("nan")
        A, B = set(a.split()), set(b.split())
        cos.append(c); jac.append(len(A & B) / max(len(A | B), 1))
        if k % 100 == 0: print(f"[7/7]   validated {k}/{len(G)}", flush=True)
    G["cosine"], G["jaccard"] = cos, jac
    G["len_ratio"] = G.out_words / G.src_words.clip(lower=1)
    G["valid"] = (G.cosine > 0.5) & (G.jaccard < 0.85) & G.len_ratio.between(0.6, 1.6)
    G.drop(columns=["text"]).to_csv("/kaggle/working/stress_generations_meta.csv", index=False)

    print(f"\n[7/7] valid {int(G.valid.sum())}/{len(G)} ({100*G.valid.mean():.1f}%)", flush=True)
    print("\n[7/7] by task x model:", flush=True)
    print(G.groupby(["task","model"]).agg(n=("valid","size"), cosine=("cosine","mean"),
          jaccard=("jaccard","mean"), len_ratio=("len_ratio","mean"),
          valid=("valid","mean")).round(3).to_string(), flush=True)
    print("\n[7/7] polish levels:", flush=True)
    pol = G[G.task=="polish"]
    if len(pol):
        print(pol.groupby("level").agg(n=("valid","size"), cosine=("cosine","mean"),
              jaccard=("jaccard","mean"), valid=("valid","mean")).round(3).to_string(), flush=True)
    print("\n[7/7] spend + time by model:", flush=True)
    print(G.groupby("model").agg(n=("cost","size"), cost=("cost","sum"),
          secs=("secs","mean")).round(3).to_string(), flush=True)
    print("\n[7/7] rule: high cosine + low jaccard = meaning kept, surface rewritten (wanted).", flush=True)
    print("[7/7]       jaccard>0.85 = barely changed | cosine<0.5 = drifted/garbled -> excluded", flush=True)
else:
    print("[7/7] nothing generated yet", flush=True)

[7/7] loaded 820 generations for validation
[7/7]   validated 100/820
[7/7]   validated 200/820
[7/7]   validated 300/820
[7/7]   validated 400/820
[7/7]   validated 500/820
[7/7]   validated 600/820
[7/7]   validated 700/820
[7/7]   validated 800/820

[7/7] valid 652/820 (79.5%)

[7/7] by task x model:
                    n  cosine  jaccard  len_ratio  valid
task     model                                          
humanize claude    82   0.757    0.394      0.879  0.963
         deepseek  82   0.911    0.677      0.930  0.915
         gemini    82   0.673    0.314      0.848  0.963
         gpt       82   0.874    0.599      0.906  0.988
         qwen      82   0.820    0.446      0.947  1.000
polish   claude    82   0.836    0.527      0.954  0.939
         deepseek  82   0.958    0.821      0.974  0.512
         gemini    82   0.815    0.549      1.007  0.744
         gpt       82   0.983    0.937      0.994  0.146
         qwen      82   0.915    0.689      0.982  0.780

[7/7] poli